# Coleta e tratamento dos dados do Deadlock

Projeto de aprendizado por reforço: cinco heróis, treze itens e vinte habilidades simuladas.

O notebook oferece duas fontes: a captura histórica da API, que reproduz os dados do modelo treinado, e uma consulta opcional à API atual. A API fornece atributos dos personagens e itens, não partidas de treinamento.

**No Colab:** envie `dados_coleta_reprodutivel.zip` à área Arquivos e execute as células na ordem. A configuração padrão usa os JSONs históricos contidos no ZIP. As quatro tabelas são reconstruídas a partir desses registros e conferidas com as assinaturas do modelo. Nenhuma pasta será criada e nenhum arquivo Python será gerado.

A coleta atual pode trazer alterações do jogo. Nesse caso, os CSVs novos terão o sufixo `_api` e não devem substituir os dados do modelo sem um novo treinamento.

## 1. Configuração

Importamos as bibliotecas utilizadas na coleta e no tratamento dos dados. Os comentários indicam a finalidade de cada uma. A maioria das bibliotecas faz parte do próprio Python; `pandas`, `requests` e `IPython` normalmente já estão disponíveis no Google Colab.

In [ ]:
import csv  # Exporta os registros do catálogo.
import io  # Manipula tabelas em memória.
import hashlib  # Calcula as assinaturas dos CSVs.
from zipfile import ZipFile  # Lê os registros históricos.
import json  # Converte estruturas aninhadas em texto para as colunas dos CSVs.
import math  # Verifica se valores numéricos são finitos.
import re  # Identifica padrões numéricos em textos.
from datetime import datetime, timezone  # Registra a data e a hora da coleta em UTC.
from pathlib import Path  # Manipula caminhos de arquivos e pastas.
from numbers import Real  # Identifica valores que são números reais.

import pandas as pd  # Organiza, filtra e exporta tabelas de dados.
import requests  # Realiza requisições HTTP à API.
from IPython.display import display  # Exibe DataFrames no notebook.

# Define a fonte dos dados e o idioma da API.
BASE_URL = "https://api.deadlock-api.com"
IDIOMA = "brazilian"
CLIENT_VERSION = None  # None consulta a versão mais recente disponível.
FONTE = "historica"  # Use "api" para consultar os dados atuais.
ARQUIVO_COLETA = Path("dados_coleta_reprodutivel.zip")

# Assinaturas SHA-256 dos quatro CSVs utilizados pelo modelo treinado.
ASSINATURAS_MODELO = {'dataset_heroes.csv': '05bcfef1548914b2513f0fd738c87402837e40b70e5260263f2837541760b327', 'dataset_itens_compraveis.csv': '9bd57971907aee4b3eee7730f9a36ff1af35d2b1de72def2e25dfd97ddd7100b', 'alcance_armas.csv': 'c46527165622911b85e800d9402bc15b8cfed59ce4906abc228731e6b3505bf6', 'habilidades_simuladas.csv': 'f2c74ed4ec95c924997edab934f1be11626c2a40aa84e64c132be1524d406e0e'}

# Escopo da versão balanceada; nomes esperados no ambiente.
HEROIS_PROJETO = {1: "Infernus", 35: "Viscoso", 3: "Vindicta", 6: "Abrams", 25: "Guarda"}
METROS_POR_UNIDADE = 0.0254  # Conversão dos alcances brutos das armas (polegadas para metros).


## 2. Extração dos dados

Os registros históricos são respostas JSON obtidas da API e armazenadas no ZIP. Na opção `api`, `consultar_api()` busca os dados diretamente dos endpoints. O filtro mantém apenas os heróis jogáveis.

In [ ]:
def consultar_api(endpoint):
    # Solicita dados em português brasileiro.
    parametros = {"language": IDIOMA}
    if CLIENT_VERSION is not None:
        parametros["client_version"] = CLIENT_VERSION
    # Consulta a API e verifica se a resposta foi bem-sucedida.
    resposta = requests.get(BASE_URL + endpoint, params=parametros, timeout=60)
    resposta.raise_for_status()
    # Converte a resposta JSON em objetos Python.
    dados = resposta.json()
    if not isinstance(dados, list) or not all(isinstance(x, dict) for x in dados):
        raise ValueError(f"Formato inesperado em {endpoint}")
    return dados


def filtrar_heroes(registros):
    # Mantém apenas heróis selecionáveis e habilitados.
    return [h for h in registros
            if h.get("player_selectable") is True and h.get("disabled") is False]


# Seleciona a origem dos registros que serão tratados.
if FONTE == "historica":
    if not ARQUIVO_COLETA.is_file():
        raise FileNotFoundError("Envie dados_coleta_reprodutivel.zip ao Colab.")
    with ZipFile(ARQUIVO_COLETA) as arquivo:
        catalogo = json.loads(arquivo.read("catalogo.json"))
        heroes = json.loads(arquivo.read("heroes_jogaveis.json"))
    origem = {"fonte": "Captura histórica da API", "idioma": IDIOMA}
elif FONTE == "api":
    catalogo = consultar_api("/v1/assets/items")
    heroes = consultar_api("/v1/assets/heroes")
    origem = {"fonte": BASE_URL, "idioma": IDIOMA,
              "data_coleta_utc": datetime.now(timezone.utc).isoformat(),
              "client_version_solicitada": CLIENT_VERSION}
else:
    raise ValueError("FONTE deve ser 'historica' ou 'api'.")

heroes_jogaveis = filtrar_heroes(heroes)
print("Origem:", origem["fonte"])
print("Registros do catálogo:", len(catalogo))
print("Heróis jogáveis:", len(heroes_jogaveis))

## 3. Inspeção das tabelas

Transformamos as listas em DataFrames e conferimos suas dimensões e os tipos de registro do catálogo: `upgrade` (itens), `weapon` (armas) e `ability` (habilidades).

In [ ]:
# Converte as listas de dicionários em tabelas do pandas.
df_catalogo = pd.DataFrame(catalogo)
df_heroes = pd.DataFrame(heroes_jogaveis)

# Confere as dimensões das tabelas e os tipos de registro.
print("Catálogo:", df_catalogo.shape)
print("Heróis jogáveis:", df_heroes.shape)
display(df_catalogo["type"].value_counts().rename("quantidade").to_frame())

# Mostra uma amostra das informações coletadas.
display(df_catalogo[["id", "name", "type"]].head())
display(df_heroes[["id", "name"]].head())

## 4. Filtragem de itens compráveis

Selecionamos os registros do tipo `upgrade` com `shopable == True`. O campo `disabled` pode estar ausente; por isso, não exigimos que ele seja explicitamente `False`. Verificamos se existem itens simultaneamente compráveis e desabilitados.

In [ ]:
# Separa os registros de itens dos demais tipos do catálogo.
upgrades = df_catalogo.loc[df_catalogo["type"].eq("upgrade")].copy()
# Usa booleanos que preservam valores ausentes.
shopable = upgrades["shopable"].astype("boolean")
disabled = upgrades.get("disabled", pd.Series(pd.NA, index=upgrades.index)).astype("boolean")

# Verifica inconsistências e mantém os itens disponíveis na loja.
assert not (shopable.fillna(False) & disabled.fillna(False)).any(), "Há itens com flags contraditórias."
compraveis = upgrades.loc[shopable.fillna(False)].copy()

# Confere a quantidade de itens e a distribuição dos preços.
print("Itens compráveis:", len(compraveis))
display(compraveis["cost"].value_counts().sort_index().rename("quantidade").to_frame())

## 5. Seleção de colunas e tratamento de propriedades

Preservamos os objetos de propriedades, descrições e efeitos em colunas JSON. Assim, o ambiente consegue consultar diretamente os bônus de cada item. Os campos ausentes permanecem vazios. A coluna `selecionado_prototipo` identifica um subconjunto de sete itens e integra o esquema do catálogo utilizado no treinamento.

In [ ]:
def ausente(valor):
    # Verifica valores ausentes sem confundir listas e dicionários com nulos.
    if isinstance(valor, (dict, list, tuple)):
        return False
    return bool(pd.isna(valor))


def serializar(valor):
    # Preserva estruturas aninhadas em texto JSON para exportação em CSV.
    return None if ausente(valor) else json.dumps(valor, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def numero(valor):
    # Converte somente números simples; conserva o motivo de não conversão.
    if ausente(valor):
        return None, "ausente"
    if isinstance(valor, bool):
        return None, "nao_numerico"
    if isinstance(valor, Real) or (isinstance(valor, str) and re.fullmatch(
            r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", valor.strip())):
        n = float(valor)
        return (n, "numerico") if math.isfinite(n) else (None, "nao_finito")
    return None, "nao_numerico_ou_com_unidade"

In [ ]:
# Escolhe as colunas úteis e traduz seus nomes quando necessário.
MAPA_ITENS = {"id": "item_id", "class_name": "class_name", "name": "nome",
              "cost": "custo", "item_slot_type": "categoria_loja", "item_tier": "tier",
              "is_active_item": "ativo", "activation": "activation", "shopable": "shopable",
              "disabled": "disabled"}
df_itens = compraveis.reindex(columns=list(MAPA_ITENS)).rename(columns=MAPA_ITENS).copy()
# Ajusta os tipos das colunas de disponibilidade.
for coluna in ["ativo", "shopable", "disabled"]:
    df_itens[coluna] = df_itens[coluna].astype("boolean")
df_itens["disabled_informado"] = df_itens["disabled"].notna()
# Mantém descrições, propriedades e componentes em JSON.
for campo in ["component_items", "description", "properties", "tooltip_sections", "imbue"]:
    df_itens[campo + "_json"] = compraveis.reindex(columns=[campo])[campo].map(serializar)
for campo in ["shop_version", "update_time"]:
    df_itens[campo] = compraveis.reindex(columns=[campo])[campo]
# Valida preços e identificadores sem descartar os dados brutos.
df_itens["custo_bruto"] = df_itens["custo"]
df_itens["custo"] = pd.to_numeric(df_itens["custo"], errors="raise")
assert df_itens["custo"].notna().all() and df_itens["custo"].gt(0).all()
for campo in ["item_id", "tier"]:
    df_itens[campo] = pd.to_numeric(df_itens[campo], errors="raise").astype("Int64")
assert df_itens["item_id"].notna().all() and df_itens["item_id"].is_unique

# Identifica o subconjunto de sete itens registrado no catálogo.
ITENS_PROTOTIPO = {"upgrade_clip_size", "upgrade_health", "upgrade_rapid_rounds",
                  "upgrade_regenerating_bullet_shield", "upgrade_magic_shield",
                  "upgrade_improved_spirit", "upgrade_endurance"}
df_itens.insert(11, "selecionado_prototipo", df_itens["class_name"].isin(ITENS_PROTOTIPO))


## 6. Atributos dos cinco heróis e de suas armas

Associamos `items.weapon_primary` ao `class_name` de um registro do tipo `weapon`. `starting_stats` fornece vida, vigor e regeneração; `weapon_info` fornece dano, cadência, carregador e tempo de recarga. A seleção usa os identificadores dos cinco heróis do ambiente e valida os nomes esperados em português. Os demais heróis permanecem na tabela de inspeção, mas não são exportados nesta versão.

In [ ]:
# Seleciona os atributos de interesse dos heróis e das armas.
CAMPOS_ARMA = ["bullet_damage", "bullets", "damage_per_shot", "shots_per_second",
               "damage_per_second", "damage_per_second_with_reload", "reload_duration", "clip_size"]
MAPA_STATS = {"max_health": "vida_base", "stamina": "vigor_base",
              "base_health_regen": "regeneracao_base"}

# Cria um índice para localizar registros pelo tipo e identificador técnico.
indice = {}
for r in catalogo:
    chave = (r.get("type"), r.get("class_name"))
    indice.setdefault(chave, []).append(r)


def resolver(tipo, classe):
    # Encontra uma referência única no catálogo e informa seu estado.
    encontrados = indice.get((tipo, classe), [])
    if not classe:
        return {}, "nao_informada"
    if not encontrados:
        return {}, "nao_encontrada"
    if len(encontrados) != 1:
        return {}, "multipla"
    return encontrados[0], "encontrada"


def stat_base(heroi, chave):
    # Extrai o valor numérico de uma estatística base do herói.
    valor = (heroi.get("starting_stats") or {}).get(chave)
    return numero(valor.get("value") if isinstance(valor, dict) else valor)[0]


# Seleciona os cinco heróis usados no ambiente.
heroes_projeto = [h for h in heroes_jogaveis if h["id"] in HEROIS_PROJETO]
assert len(heroes_projeto) == len(HEROIS_PROJETO)
assert {h["id"]: h["name"] for h in heroes_projeto} == HEROIS_PROJETO, "Heróis ausentes ou renomeados: revise o ambiente."

# Associa cada herói à sua arma principal e extrai os atributos.
linhas_heroes = []
for h in heroes_projeto:
    ref = (h.get("items") or {}).get("weapon_primary")
    arma, status = resolver("weapon", ref)
    info = arma.get("weapon_info") or {}
    linhas_heroes.append({"hero_id": h["id"], "nome_heroi": h["name"],
                          "weapon_id": arma.get("id"), "weapon_primary": ref,
                          "status_referencia": status,
                          **{coluna: stat_base(h, k) for k, coluna in MAPA_STATS.items()},
                          **{k: numero(info.get(k))[0] for k in CAMPOS_ARMA},
                          **{k: h.get(k) for k in ["player_selectable", "disabled", "development_state", "in_development"]}})
# Confere identificadores, referências e valores ausentes.
df_heroes_final = pd.DataFrame(linhas_heroes)
assert df_heroes_final["hero_id"].is_unique
assert df_heroes_final["status_referencia"].eq("encontrada").all(), "Revise as referências de armas."
assert df_heroes_final[list(MAPA_STATS.values()) + CAMPOS_ARMA].notna().all().all(), "Revise atributos ausentes."
# Inspeciona os atributos e a distribuição da vida base.
print("Valores distintos de weapon_power:", sorted({stat_base(h, "weapon_power") for h in heroes_projeto}, key=str))
display(df_heroes_final[["nome_heroi", "vida_base", "vigor_base", "regeneracao_base", "bullet_damage", "shots_per_second"]].head())
display(df_heroes_final["vida_base"].describe().to_frame())
# Vida, vigor, quantidade de projéteis e munição são valores inteiros.
for coluna in ["vida_base", "vigor_base", "bullets", "clip_size"]:
    if not df_heroes_final[coluna].mod(1).eq(0).all():
        raise ValueError(f"Valor não inteiro em {coluna}")
    df_heroes_final[coluna] = df_heroes_final[coluna].astype(int)


## 7. Alcance, dispersão e recarga das armas

O ambiente balanceado usa a distância e a dispersão (`spread`) para estimar acertos. Os alcances de `weapon_info` são convertidos para metros com o fator 0,0254, seguindo a conversão da versão histórica. A recarga por bala e o atraso inicial são preservados: o Abrams, por exemplo, não recarrega todo o carregador em uma única unidade de `reload_duration`. A fórmula da probabilidade de acerto pertence ao ambiente, não à API.

In [ ]:
def numero_obrigatorio(valor, contexto):
    n, _ = numero(valor)
    if n is None:
        raise ValueError(f"Valor numérico inválido ou ausente: {contexto}: {valor!r}")
    return n


linhas_alcance = []
# Mantém a ordem usada pelo ambiente para facilitar a conferência.
por_id = {h["id"]: h for h in heroes_projeto}
for hero_id, nome in HEROIS_PROJETO.items():
    ref = por_id[hero_id]["items"]["weapon_primary"]
    arma, status = resolver("weapon", ref)
    if status != "encontrada":
        raise ValueError(f"Arma não localizada: {nome}/{ref}")
    info = arma["weapon_info"]

    def campo(chave):
        return numero_obrigatorio(info.get(chave), f"{ref}/{chave}")

    recarga_por_bala = info.get("reload_single_bullets", False)
    if not isinstance(recarga_por_bala, bool):
        raise ValueError(f"Modo de recarga inválido: {ref}")
    atraso = campo("reload_single_bullets_initial_delay") if recarga_por_bala else 0.0
    linhas_alcance.append({
        "heroi": nome, "class_name": ref,
        "alcance_max_m": campo("range") * METROS_POR_UNIDADE,
        "queda_inicio_m": campo("damage_falloff_start_range") * METROS_POR_UNIDADE,
        "queda_fim_m": campo("damage_falloff_end_range") * METROS_POR_UNIDADE,
        "escala_inicio": campo("damage_falloff_start_scale"),
        "escala_fim": campo("damage_falloff_end_scale"),
        "spread": campo("spread"), "recarga_por_bala": recarga_por_bala,
        "recarga_inicial_s": atraso
    })

df_alcances = pd.DataFrame(linhas_alcance)
assert df_alcances.heroi.is_unique
assert (df_alcances.queda_inicio_m <= df_alcances.queda_fim_m).all()
assert (df_alcances.queda_fim_m <= df_alcances.alcance_max_m).all()
assert (df_alcances[["queda_inicio_m", "spread", "recarga_inicial_s"]] >= 0).all().all()
assert (df_alcances.escala_inicio >= df_alcances.escala_fim).all()
display(df_alcances)

# Arredonda apenas a representação decimal dos alcances em metros.
colunas_metros = ["alcance_max_m", "queda_inicio_m", "queda_fim_m"]
df_alcances[colunas_metros] = df_alcances[colunas_metros].round(8)
if df_alcances["escala_inicio"].mod(1).eq(0).all():
    df_alcances["escala_inicio"] = df_alcances["escala_inicio"].astype(int)


## 8. Extração e tratamento das 20 habilidades

Cada herói referencia suas quatro habilidades por `items.signature1` até `items.signature4`. A referência é resolvida no catálogo de `ability`. As propriedades de cada habilidade têm nomes diferentes: por exemplo, `Damage` é dano de impacto, `DPS` é dano por segundo, e `CombatBarrier` é barreira. A escala espiritual é extraída de `scale_function.stat_scale` da propriedade correspondente.

O mapeamento abaixo preserva a ordem e a semântica das habilidades do ambiente v6.0 (a ordem afeta a prioridade de ativação e a infusão de Compressão de Recarga). Alcances de projétil ausentes são aproximados em 25 m; Bola de Gosma usa raio de colisão + 0,5 m e dois segundos de uso tático; o máximo de acúmulos do Incendiar é 100. Esses valores e as regras de uso não são extraídos como fatos da API. As colunas `fonte_*` e `nota` documentam a distinção.

**Aproximação adicional:** o Soco Pegajoso tem coeficiente associado a dano leve corpo a corpo (`ELightMeleeDamage`) no catálogo histórico. Para reproduzir o ambiente v6.0, esse coeficiente é tratado como escala espiritual no simulador; não é uma escala espiritual extraída diretamente da API.

In [ ]:
# Regras de interpretação da API; cada entrada define uma habilidade.
# Números literais são aproximações do simulador.
MAPA_HABILIDADES = [{'heroi': 'Abrams',
  'slot': 2,
  'class_name': 'citadel_ability_bull_charge',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 'ChargeRadius'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'ChargeRadius',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Contato próximo; sem deslocamento e atordoamento.'},
 {'heroi': 'Abrams',
  'slot': 1,
  'class_name': 'citadel_ability_bull_heal',
  'tipo': 'continuo',
  'campos': {'dano': 'DPS',
             'roubo_vida': 'HealingFactor',
             'alcance_m': 'Radius',
             'duracao_s': 'AbilityDuration'},
  'fonte_dano': 'DPS',
  'fonte_cura': '',
  'fonte_alcance': 'Radius',
  'fonte_duracao': 'AbilityDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': 'HealingFactor',
  'nota': 'Aura de dano e cura proporcional ao dano efetivo.'},
 {'heroi': 'Guarda',
  'slot': 1,
  'class_name': 'ability_warden_crowd_control',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 25},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'aproximado',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Alcance de projétil aproximado; sem redução de atributos.'},
 {'heroi': 'Guarda',
  'slot': 3,
  'class_name': 'ability_warden_lock_down',
  'tipo': 'atrasado',
  'campos': {'dano': 'Damage', 'alcance_m': 'AbilityCastRange', 'atraso_s': 'EscapeTime'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'AbilityCastRange',
  'fonte_duracao': '',
  'fonte_atraso': 'EscapeTime',
  'fonte_roubo_vida': '',
  'nota': 'Dano após atraso, ponderado pela chance estimada de o alvo não escapar.'},
 {'heroi': 'Guarda',
  'slot': 2,
  'class_name': 'ability_warden_high_alert',
  'tipo': 'barreira',
  'campos': {'cura': 'CombatBarrier', 'duracao_s': 'AbilityDuration'},
  'fonte_dano': '',
  'fonte_cura': 'CombatBarrier',
  'fonte_alcance': 'pessoal/passiva',
  'fonte_duracao': 'AbilityDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Barreira temporária, sem velocidade adicional.'},
 {'heroi': 'Infernus',
  'slot': 1,
  'class_name': 'ability_incendiary_projectile',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 'AbilityCastRange'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'AbilityCastRange',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Somente dano direto do Napalm.'},
 {'heroi': 'Infernus',
  'slot': 2,
  'class_name': 'ability_flame_dash',
  'tipo': 'canal',
  'campos': {'dano': 'DPS', 'alcance_m': 'FlameAuraRadius', 'duracao_s': 'AbilityDuration'},
  'fonte_dano': 'DPS',
  'fonte_cura': '',
  'fonte_alcance': 'FlameAuraRadius',
  'fonte_duracao': 'AbilityDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Aura ao redor do herói, sem deslocamento; não dispara durante o avanço.'},
 {'heroi': 'Infernus',
  'slot': 3,
  'class_name': 'ability_afterburn',
  'tipo': 'incendiar',
  'campos': {'dano': 'DPS',
             'duracao_s': 'BurnDuration',
             'acumulo_por_bala': 'BuildUpBulletPercentPerHit',
             'max_acumulos': 100,
             'janela_acumulo_s': 'BuildUpDuration'},
  'fonte_dano': 'DPS',
  'fonte_cura': '',
  'fonte_alcance': 'pessoal/passiva',
  'fonte_duracao': 'BurnDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Acúmulo por bala; sem críticos, sem renovação por outras habilidades.'},
 {'heroi': 'Vindicta',
  'slot': 2,
  'class_name': 'citadel_ability_hornet_leap',
  'tipo': 'voo',
  'campos': {'dano': 'MagicDamagePerBullet', 'duracao_s': 'AbilityDuration'},
  'fonte_dano': 'MagicDamagePerBullet',
  'fonte_cura': '',
  'fonte_alcance': 'pessoal',
  'fonte_duracao': 'AbilityDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Dano espiritual por bala acertada durante o voo; altitude aproximada de 6 m; sem bônus '
          'de alcance de itens. Ativação tática apenas a partir de 8 m.'},
 {'heroi': 'Vindicta',
  'slot': 1,
  'class_name': 'citadel_ability_hornet_chain',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 25},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'aproximado',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Alcance de projétil aproximado; sem amarras.'},
 {'heroi': 'Vindicta',
  'slot': 3,
  'class_name': 'citadel_ability_hornet_sting',
  'tipo': 'corvo',
  'campos': {'dano': 'ImpactDamage',
             'alcance_m': 25,
             'duracao_s': 'DebuffDuration',
             'sangramento_pct_s': 'DotHealthPercent'},
  'fonte_dano': 'ImpactDamage',
  'fonte_cura': '',
  'fonte_alcance': 'aproximado',
  'fonte_duracao': 'DebuffDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Impacto e sangramento proporcional à vida atual; dano contínuo esperado. Sem redução de '
          'resistências.'},
 {'heroi': 'Vindicta',
  'slot': 4,
  'class_name': 'citadel_ability_hornet_snipe',
  'tipo': 'execucao',
  'campos': {'dano': 'Damage',
             'alcance_m': 'Range',
             'atraso_s': 'TimeToFullCharge',
             'bonus_execucao': 'LowHealthEnemyDamageBonus',
             'cargas': 'AbilityCharges',
             'intervalo_cargas_s': 'AbilityCooldownBetweenCharge'},
  'fonte_dano': 'Damage + LowHealthEnemyDamageBonus',
  'fonte_cura': '',
  'fonte_alcance': 'Range',
  'fonte_duracao': '',
  'fonte_atraso': 'TimeToFullCharge',
  'fonte_roubo_vida': '',
  'nota': 'Usada contra alvo abaixo de 50%; carrega sem disparar arma; bônus cresce linearmente '
          'até a vida zero. Sem bônus persistente por abate.'},
 {'heroi': 'Viscoso',
  'slot': 1,
  'class_name': 'viscous_goo_grenade',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 25},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'aproximado',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Alcance de projétil aproximado; uma explosão.'},
 {'heroi': 'Viscoso',
  'slot': 3,
  'class_name': 'viscous_telepunch',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 'AbilityCastRange'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'AbilityCastRange',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Um impacto, sem deslocamento.'},
 {'heroi': 'Viscoso',
  'slot': 2,
  'class_name': 'viscous_restorative_goo',
  'tipo': 'cubo',
  'campos': {'cura': 'BonusHealthRegen', 'duracao_s': 'AbilityDuration'},
  'fonte_dano': '',
  'fonte_cura': 'BonusHealthRegen',
  'fonte_alcance': 'pessoal/passiva',
  'fonte_duracao': 'AbilityDuration',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Autocura por segundo e proteção; não ataca enquanto protegido.'},
 {'heroi': 'Infernus',
  'slot': 4,
  'class_name': 'ability_fire_bomb',
  'tipo': 'atrasado',
  'campos': {'dano': 'Damage', 'alcance_m': 'Radius', 'atraso_s': 'ExplodeDelay'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'Radius',
  'fonte_duracao': '',
  'fonte_atraso': 'ExplodeDelay',
  'fonte_roubo_vida': '',
  'nota': 'Dano esperado após atraso; sem atordoamento.'},
 {'heroi': 'Viscoso',
  'slot': 4,
  'class_name': 'viscous_goo_bowling_ball',
  'tipo': 'bola',
  'campos': {'dano': 'Damage',
             'alcance_m': 'BallHitRadius',
             'duracao_s': 2,
             'resistencia_ativa': 'TechResist'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'BallHitRadius + 0.5 m (aproximação)',
  'fonte_duracao': '2 s de uso tático (aproximação)',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Uma colisão, resistência durante 2 s de rolamento, sem disparar; cancela cedo. Sem '
          'movimento real.'},
 {'heroi': 'Abrams',
  'slot': 3,
  'class_name': 'citadel_ability_passive_beefy',
  'tipo': 'resiliencia',
  'campos': {'cura': 'BonusHealthRegen',
             'regen_dano': 'RegenIncomingDamagePercent',
             'regen_duracao_s': 'RegenIncomingDamageDuration'},
  'fonte_dano': '',
  'fonte_cura': 'BonusHealthRegen + RegenIncomingDamagePercent',
  'fonte_alcance': '',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Regeneração base adicional e reserva de cura exponencial em 20 s, aproximando '
          'regeneração gradual do dano.'},
 {'heroi': 'Abrams',
  'slot': 4,
  'class_name': 'citadel_ability_bull_leap',
  'tipo': 'direto',
  'campos': {'dano': 'Damage', 'alcance_m': 'ImpactRadius'},
  'fonte_dano': 'Damage',
  'fonte_cura': '',
  'fonte_alcance': 'ImpactRadius',
  'fonte_duracao': '',
  'fonte_atraso': '',
  'fonte_roubo_vida': '',
  'nota': 'Impacto próximo; sem deslocamento ou atordoamento.'},
 {'heroi': 'Guarda',
  'slot': 4,
  'class_name': 'ability_warden_riot_protocol',
  'tipo': 'ritual',
  'campos': {'dano': 'PulseDPS',
             'alcance_m': 'Radius',
             'roubo_vida': 'HealthStealPctHero',
             'duracao_s': 'AbilityDuration',
             'atraso_s': 'AbilityCastDelay',
             'resistencia_ativa': 'TechResist'},
  'fonte_dano': 'PulseDPS',
  'fonte_cura': '',
  'fonte_alcance': 'Radius',
  'fonte_duracao': 'AbilityDuration',
  'fonte_atraso': 'AbilityCastDelay',
  'fonte_roubo_vida': 'HealthStealPctHero',
  'nota': 'Canalização após 2 s de preparação, resistências e roubo de vida; sem bônus de '
          'imobilização.'}]


In [ ]:
CAMPOS_HABILIDADE = [
    "dano", "escala_dano", "cura", "escala_cura", "roubo_vida", "alcance_m",
    "duracao_s", "atraso_s", "cooldown_s", "acumulo_por_bala", "max_acumulos",
    "janela_acumulo_s", "bonus_execucao", "escala_execucao", "sangramento_pct_s",
    "cargas", "intervalo_cargas_s", "resistencia_ativa", "regen_dano", "regen_duracao_s"
]
PERCENTUAIS = {"roubo_vida", "sangramento_pct_s", "resistencia_ativa", "regen_dano"}
ESCALAS = {"dano": "escala_dano", "cura": "escala_cura", "bonus_execucao": "escala_execucao"}


def propriedade(props, chave):
    # Não transforma propriedades ausentes em zero.
    meta = props.get(chave)
    if not isinstance(meta, dict) or "value" not in meta:
        raise ValueError(f"Propriedade ausente ou inválida: {chave}")
    return meta


def valor_propriedade(props, chave, em_metros=False):
    valor = propriedade(props, chave)["value"]
    if em_metros:
        # As propriedades de alcance das habilidades trazem a unidade m.
        if not isinstance(valor, str) or not re.fullmatch(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)\s*m", valor.strip()):
            raise ValueError(f"Unidade de alcance inesperada em {chave}: {valor!r}")
        valor = valor.strip()[:-1].strip()
    return numero_obrigatorio(valor, chave)


def escala_espiritual(props, chave, classe):
    escala = propriedade(props, chave).get("scale_function") or {}
    # Soco Pegajoso escala com dano leve corpo a corpo na API; o motor
    # v6.0 aproxima esse coeficiente como escala espiritual.
    if classe == "viscous_telepunch" and chave == "Damage" and escala.get("specific_stat_scale_type") == "ELightMeleeDamage":
        return numero_obrigatorio(escala.get("stat_scale"), f"escala de {chave}")
    if (escala.get("specific_stat_scale_type") != "ETechPower"
            and escala.get("class_name") not in {"scale_function_tech_damage", "scale_function_healing_spirit_scale"}):
        raise ValueError(f"Escala espiritual inesperada em {chave}")
    return numero_obrigatorio(escala.get("stat_scale"), f"escala de {chave}")


por_nome = {h["name"]: h for h in heroes_projeto}
linhas_habilidades = []
for regra in MAPA_HABILIDADES:
    heroi, slot = regra["heroi"], regra["slot"]
    classe = por_nome[heroi]["items"].get(f"signature{slot}")
    if classe != regra["class_name"]:
        raise ValueError(f"Referência de habilidade alterada: {heroi}/{slot}: {classe}")
    habilidade, status = resolver("ability", classe)
    if status != "encontrada":
        raise ValueError(f"Habilidade não encontrada: {classe}")
    props = habilidade["properties"]

    # Zero significa efeito não representado, não uma propriedade ausente da API.
    linha = {c: 0.0 for c in CAMPOS_HABILIDADE}
    linha["cargas"] = 1  # Uma carga no motor simplificado, salvo Assassinar.
    linha["cooldown_s"] = valor_propriedade(props, "AbilityCooldown")
    for campo, fonte in regra["campos"].items():
        if isinstance(fonte, str):
            valor = valor_propriedade(props, fonte, em_metros=(campo == "alcance_m"))
            linha[campo] = valor / 100 if campo in PERCENTUAIS else valor
            # A regeneração passiva do Abrams não escala com poder espiritual.
            if campo in ESCALAS and not (regra["tipo"] == "resiliencia" and campo == "cura"):
                linha[ESCALAS[campo]] = escala_espiritual(props, fonte, classe)
        else:
            linha[campo] = float(fonte)

    if regra["tipo"] == "bola":
        linha["alcance_m"] += 0.5  # Margem aproximada de colisão.
    if regra["tipo"] in ("bola", "ritual"):
        if valor_propriedade(props, "BulletResist") != valor_propriedade(props, "TechResist"):
            raise ValueError(f"Resistências diferentes: revise o motor para {classe}")
    if regra["tipo"] == "execucao" and valor_propriedade(props, "LowHealthEnemyThresholdPct") != 50:
        raise ValueError("Limiar de Assassinar alterado: revise o motor de combate.")
    if linha["cargas"] < 1 or not float(linha["cargas"]).is_integer():
        raise ValueError(f"Número de cargas inválido: {classe}")
    linha["cargas"] = int(linha["cargas"])
    linhas_habilidades.append({
        "heroi": heroi, "slot": slot, "nome": habilidade["name"],
        "class_name": classe, "tipo": regra["tipo"], **linha,
        **{k: v for k, v in regra.items() if k.startswith("fonte_") or k == "nota"}
    })

# Mesmas colunas e mesma ordem de prioridade do ambiente v6.0.
ORDEM_HABILIDADES = ["heroi", "slot", "nome", "class_name", "tipo"] + CAMPOS_HABILIDADE[:12] + [
    "fonte_dano", "fonte_cura", "fonte_alcance", "fonte_duracao", "fonte_atraso", "fonte_roubo_vida", "nota"
] + CAMPOS_HABILIDADE[12:]
df_habilidades = pd.DataFrame(linhas_habilidades)[ORDEM_HABILIDADES]
assert len(df_habilidades) == 20
assert not df_habilidades.duplicated(["heroi", "slot"]).any()
assert set(df_habilidades.heroi) == set(HEROIS_PROJETO.values())
assert (df_habilidades.groupby("heroi").slot.agg(set) == {1, 2, 3, 4}).all()
assert df_habilidades[CAMPOS_HABILIDADE].map(lambda x: math.isfinite(float(x)) and x >= 0).all().all()
display(df_habilidades[["heroi", "slot", "nome", "tipo", "dano", "escala_dano", "alcance_m", "cooldown_s"]])

# Evita artefatos de representação binária na conversão do percentual.
df_habilidades["sangramento_pct_s"] = df_habilidades["sangramento_pct_s"].round(10)


## 9. Validação dos 13 itens utilizados no ambiente

O catálogo permanece completo, mas conferimos se os 13 itens passivos selecionados pelo simulador ainda existem com as propriedades numéricas necessárias. Uma mudança de esquema, custo ou limiar deve ser analisada antes de retreinar. O saldo do ambiente usa passos de 800 almas e teto de 3.200.

In [ ]:
# Propriedades usadas nas regras de itens do ambiente balanceado.
PROPRIEDADES_EXIGIDAS = {'upgrade_health': ['BonusHealth'],
 'upgrade_endurance': ['BonusHealthRegen', 'OutOfCombatHealthRegen'],
 'upgrade_improved_spirit': ['TechPower'],
 'upgrade_rapid_rounds': ['BonusFireRate'],
 'upgrade_clip_size': ['BaseAttackDamagePercent', 'BonusClipSizePercent'],
 'upgrade_regenerating_bullet_shield': ['BulletResist',
                                        'OutOfCombatHealthRegen',
                                        'BaseAttackDamagePercent',
                                        'BonusFireRate'],
 'upgrade_magic_shield': ['TechResist', 'OutOfCombatHealthRegen', 'TechPower', 'CooldownReduction'],
 'upgrade_vampire': ['BonusHealth', 'BaseAttackDamagePercent', 'BulletLifestealPercent'],
 'upgrade_health_stealing_magic': ['BonusHealth', 'TechPower', 'AbilityLifestealPercentHero'],
 'upgrade_healing_booster': ['OutOfCombatHealthRegen', 'HealAmpCastPercent', 'HealAmpRegenPercent'],
 'upgrade_magic_tempo': ['CooldownReduction'],
 'upgrade_improved_bullet_armor': ['BulletResist',
                                   'OutOfCombatHealthRegen',
                                   'BulletResistBelowThreshold'],
 'upgrade_tech_purge': ['TechResist', 'OutOfCombatHealthRegen', 'TechResistBelowThreshold']}
LIMIARES = {'upgrade_regenerating_bullet_shield': {'LifeThreshold': 65},
 'upgrade_magic_shield': {'LifeThreshold': 65},
 'upgrade_improved_bullet_armor': {'HealthThreshold': 50},
 'upgrade_tech_purge': {'HealthThreshold': 50}}

assert df_itens.class_name.is_unique, "Classes de itens duplicadas."
selecionados = df_itens.set_index("class_name").loc[list(PROPRIEDADES_EXIGIDAS)]
assert selecionados.activation.eq("passive").all()
assert selecionados.shopable.eq(True).all()
assert selecionados.custo.isin([800, 1600, 2400, 3200]).all(), "Preços incompatíveis com o espaço de estados atual."
for classe, campos in PROPRIEDADES_EXIGIDAS.items():
    props = json.loads(selecionados.loc[classe, "properties_json"])
    for campo in campos:
        valor_propriedade(props, campo)
    for chave, esperado in LIMIARES.get(classe, {}).items():
        if valor_propriedade(props, chave) != esperado:
            raise ValueError(f"Limiar alterado: {classe}/{chave}")
display(selecionados[["nome", "custo"]])


## 10. Exportação e validação

As quatro tabelas são serializadas com as mesmas convenções dos dados do modelo: UTF-8 com BOM, ordem de colunas e formatação de números e JSON. A assinatura SHA-256 é calculada como no ambiente, normalizando as quebras de linha.

No modo histórico, se uma biblioteca representar um número de modo ligeiramente diferente, comparamos toda a tabela com a referência autenticada. Somente se os registros forem equivalentes em todas as colunas, preservamos a serialização de referência. Se houver divergência de conteúdo, a exportação é interrompida. No modo API, dados incompatíveis são gravados com sufixo `_api` para não substituir o conjunto treinado.

In [ ]:
# Exporta os itens mantendo a representação dos campos do catálogo.
def exportar_itens():
    campos = list(df_itens.columns)
    mapa = dict(zip(MAPA_ITENS.values(), MAPA_ITENS.keys()))
    mapa.update({"shop_version": "shop_version", "update_time": "update_time",
                 "custo_bruto": "cost"})
    texto = io.StringIO(newline="")
    escritor = csv.DictWriter(texto, fieldnames=campos, lineterminator="\r\n")
    escritor.writeheader()
    for registro in catalogo:
        if registro.get("type") != "upgrade" or registro.get("shopable") is not True:
            continue
        linha = {k: registro.get(v) for k, v in mapa.items()}
        linha["disabled_informado"] = registro.get("disabled") is not None
        linha["selecionado_prototipo"] = registro["class_name"] in ITENS_PROTOTIPO
        for campo in ["component_items", "description", "properties", "tooltip_sections", "imbue"]:
            linha[campo + "_json"] = serializar(registro.get(campo))
        # Escreve booleanos em minúsculas e preserva os números da API.
        linha = {k: str(v).lower() if isinstance(v, bool) else v for k, v in linha.items()}
        escritor.writerow(linha)
    return texto.getvalue().encode("utf-8-sig")


# Padroniza a conversão dos valores numéricos da tabela de heróis.
herois_exportacao = pd.read_csv(io.StringIO(df_heroes_final.to_csv(index=False)),
                                float_precision="high")

def serializar_csv(tabela):
    return tabela.to_csv(index=False, lineterminator="\n").encode("utf-8-sig")


# Calcula a mesma assinatura usada para carregar a Tabela Q.
def assinatura(conteudo):
    texto = conteudo.decode("utf-8").replace("\r\n", "\n").replace("\r", "\n")
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def tabelas_equivalentes(a, b):
    # Compara todos os campos, aceitando apenas diferenças de serialização.
    x = pd.read_csv(io.BytesIO(a), encoding="utf-8-sig", float_precision="round_trip")
    y = pd.read_csv(io.BytesIO(b), encoding="utf-8-sig", float_precision="round_trip")
    if x.shape != y.shape or list(x.columns) != list(y.columns):
        return False
    for coluna in x.columns:
        if coluna.endswith("_json"):
            for v, w in zip(x[coluna], y[coluna]):
                if pd.isna(v) and pd.isna(w):
                    continue
                if pd.isna(v) or pd.isna(w) or json.loads(v) != json.loads(w):
                    return False
        else:
            try:
                pd.testing.assert_series_equal(x[coluna], y[coluna], check_dtype=False,
                                               check_exact=False, rtol=1e-14, atol=1e-14)
            except AssertionError:
                return False
    return True


# Prepara os quatro CSVs sem criar pastas adicionais.
conteudos = {
    "dataset_heroes.csv": serializar_csv(herois_exportacao),
    "dataset_itens_compraveis.csv": exportar_itens(),
    "alcance_armas.csv": serializar_csv(df_alcances),
    "habilidades_simuladas.csv": serializar_csv(df_habilidades)
}

if FONTE == "historica":
    # Confere o conteúdo antes de gravar qualquer arquivo.
    with ZipFile(ARQUIVO_COLETA) as arquivo:
        for nome, conteudo in conteudos.items():
            referencia = arquivo.read(nome)
            esperada = ASSINATURAS_MODELO[nome]
            if assinatura(referencia) != esperada:
                raise ValueError(f"Referência com assinatura inválida: {nome}")
            if conteudo != referencia:
                if not tabelas_equivalentes(conteudo, referencia):
                    raise ValueError(f"Dados reconstruídos diferentes da referência: {nome}")
                # Preserva os bytes autenticados quando apenas o formato diverge.
                conteudos[nome] = referencia
                print(f"{nome}: serialização de referência preservada.")
            if assinatura(conteudos[nome]) != esperada:
                raise ValueError(f"Assinatura incompatível: {nome}")

compativel = all(assinatura(v) == ASSINATURAS_MODELO[k] for k, v in conteudos.items())
resumo = []
for nome, conteudo in conteudos.items():
    # Dados incompatíveis da API não sobrescrevem os CSVs usados pelo modelo.
    destino = Path(nome if compativel else nome.replace(".csv", "_api.csv"))
    destino.write_bytes(conteudo)
    leitura = pd.read_csv(destino, encoding="utf-8-sig")
    resumo.append({"arquivo": destino.name, "linhas": len(leitura),
                   "colunas": len(leitura.columns),
                   "assinatura_compativel": assinatura(conteudo) == ASSINATURAS_MODELO[nome]})

display(pd.DataFrame([origem]))
display(pd.DataFrame(resumo))
if compativel:
    print("Os quatro CSVs são compatíveis com o modelo treinado.")
else:
    print("Dados diferentes do modelo: revise os atributos e retreine antes de utilizá-los.")

### Conclusão

Os registros da API são transformados em quatro tabelas usadas pelo ambiente. A conferência de assinatura garante que o treinamento, a avaliação e o chatbot consultem o mesmo conjunto de dados. As regras de combate, a precisão aproximada e a função de recompensa são decisões de modelagem do simulador.